<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/notebooks/02_train_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · 2. QLoRA fine-tuning of Mistral-7B-Instruct

**Runtime:** GPU (T4 is enough). Runtime → Change runtime type → T4 GPU.
Trains on up to 2,000 releases for one epoch: roughly 3-4 hours on a T4 (a rough estimate). Checkpoints go to
Google Drive every 50 steps; if the session drops, re-run the setup cells and the training cell with `--resume`.
Kaggle notebooks (free T4, 12-hour sessions) are a good alternative.

Needs the dataset from notebook 1 on your Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo and install dependencies (~2 min)
REPO_BRANCH = "main"   # to try a branch before merging it, put its name here
!git clone -q -b $REPO_BRANCH https://github.com/PujaSridhar/finsight /content/finsight 2>/dev/null || (cd /content/finsight && git pull -q)
%cd /content/finsight
!pip install -q -r requirements.txt

In [ ]:
# Keep data, checkpoints and results on Google Drive so a disconnect doesn't lose them
# and every notebook sees the same dataset.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/finsight/outputs /content/drive/MyDrive/finsight/results
!mkdir -p /content/drive/MyDrive/finsight/data
!rm -rf outputs results data && ln -s /content/drive/MyDrive/finsight/outputs outputs && ln -s /content/drive/MyDrive/finsight/results results && ln -s /content/drive/MyDrive/finsight/data data

In [ ]:
# Mistral-7B-Instruct-v0.3 is gated: accept its terms at
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3, then paste a read token here
# (or store it as the Colab secret HF_TOKEN).
from huggingface_hub import login
login()

In [ ]:
!wc -l data/earnings/*.jsonl

## 10-minute smoke test
Catches setup problems (token, GPU memory, library versions) before the long run.

In [ ]:
!python -m finsight.train --config configs/earnings_smoke.yaml

## Train

In [ ]:
!python -m finsight.train --config configs/earnings_t4.yaml
# after a disconnect:
# !python -m finsight.train --config configs/earnings_t4.yaml --resume

In [ ]:
import json
from IPython.display import Image, display
s = json.load(open("outputs/finsight-earnings/training_summary.json"))
print({k: v for k, v in s.items() if k != "log_history"})
display(Image("results/loss_curve_earnings.png"))

## Optional ablation: FinQA / TAT-QA warm-up first

Trains a numerical-reasoning adapter on 4,000 verified FinQA / TAT-QA examples (about 2-3 more hours), then
continues it on earnings. Keep it only if it beats the plain run on the earnings test set (notebook 3).

In [ ]:
# !python -m finsight.data --out data/processed
# !python -m finsight.train --config configs/qlora_t4.yaml
# !sed -e 's#^init_adapter: null.*#init_adapter: outputs/finsight-mistral-7b-qlora/final#' \
#      -e 's#^output_dir: .*#output_dir: outputs/finsight-earnings-warmup#' configs/earnings_t4.yaml > configs/earnings_warmup.yaml
# !python -m finsight.train --config configs/earnings_warmup.yaml

The adapter is in `outputs/finsight-earnings/final`. Next: `03_compare.ipynb`.